In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

pd.set_option('display.float_format', '{:.2f}'.format)

In [13]:
df = pd.read_parquet('/Users/kaungkhantkyaw/Documents/GitHub/Analyse-US-Domestic-flights/Data/Cleaned/DB1B_Market_2025_1_cleaned.parquet')

print(df.shape)
display(df.columns)


(7290528, 23)


Index(['ItinID', 'MktID', 'MktCoupons', 'Year', 'Quarter',
       'OriginCityMarketID', 'Origin', 'OriginState', 'DestCityMarketID',
       'Dest', 'DestState', 'TkCarrierChange', 'OpCarrierChange', 'RPCarrier',
       'TkCarrier', 'OpCarrier', 'Passengers', 'MktFare', 'MktDistance',
       'MktDistanceGroup', 'NonStopMiles', 'MktGeoType', 'fare_group'],
      dtype='str')

In [14]:
print('Invalid fares: ', (df['MktFare'] <= 0).sum())
print('Invalid distance: ', (df['MktDistance'] <= 0).sum())
print('Missing values: ', df.isnull().sum().sum())

#sanity check

Invalid fares:  0
Invalid distance:  0
Missing values:  0


In [15]:
#reapplying the filters from notebook 2
low_fare = df['MktFare'].quantile(0.01)
high_fare = df['MktFare'].quantile(0.99)
print("Before trimming: ",df.shape)

df = df[(df['MktFare'] >= low_fare) & (df['MktFare'] <= high_fare)].copy()
df['Route'] = df['Origin'] + '-' + df['Dest']
print("After trimming: ",df.shape)

Before trimming:  (7290528, 23)
After trimming:  (7189747, 24)


Columns to Drop

ItinID, MktID: Identifiers that do not add value to the model
Year, Quater: Currently all data is in 2025 and in Q1, thus these add no value until we add more Years and Quaters
OriginCityMarketID, DestCityMarketID: Airport Codes, these codes provide the same values given by Origin and Dest, thus we will drop it
OriginState, DestState: Might drop in second run to see how the model fares as technically the origin and dest already provide state information
RPCarrier, OpCarrier: TKCarrier is sufficient
fare_group: EDA column from notebook 1
NonStopMiles: Drop after creating extra_miles

In [18]:
#Computing this before dropping NonStopMiles
df['extra_miles'] = df['MktDistance'] -df['NonStopMiles']
df['carrier_mismatch'] = (df['TkCarrier'] != df['OpCarrier']).astype(int)

In [ ]:
df['log_passengers'] = np.log1p(df['Passengers']) #log transformation used as data is right skewed, log1p used to ensure reproducibility
df['distance_per_coupon'] = df['MktDistance'] / df['MktCoupons']

In [24]:
drop_cols = ["ItinID", "MktID", "Year", "Quarter", "OriginCityMarketID", "DestCityMarketID", "RPCarrier", "OpCarrier", "fare_group", "NonStopMiles", "TkCarrierChange", "OpCarrierChange"]
df = df.drop(columns = [c for c in drop_cols if c in df.columns])
print(df.shape)
print(df.columns)

(7189747, 16)
Index(['MktCoupons', 'Origin', 'OriginState', 'Dest', 'DestState', 'TkCarrier',
       'Passengers', 'MktFare', 'MktDistance', 'MktDistanceGroup',
       'MktGeoType', 'Route', 'extra_miles', 'carrier_mismatch',
       'log_passengers', 'distance_per_coupon'],
      dtype='str')


In [30]:
target = 'MktFare'

feature_cols = ['Origin', 'Dest', 'OriginState', 'DestState', 'TkCarrier', 'Passengers', 
'MktDistance', 'MktCoupons', 'MktDistanceGroup', 'MktGeoType', 'extra_miles', 
'carrier_mismatch', 'log_passengers', 'distance_per_coupon']
#route is dropped as it gives the same information as Origin and Dest, it can be used potentially in the future

#check if all features are present
missing_cols = [c for c in feature_cols if c not in df.columns]
print("missing features: ", missing_cols)

df_model = df[feature_cols + [target]].copy()

missing features:  []


In [31]:
out_path = '/Users/kaungkhantkyaw/Documents/GitHub/Analyse-US-Domestic-flights/Data/Cleaned/DB1B_Market_model_ready.parquet'
df_model.to_parquet(out_path, index = False)

print("Saved: ", out_path)
print("Shape: ", df_model.shape)

Saved:  /Users/kaungkhantkyaw/Documents/GitHub/Analyse-US-Domestic-flights/Data/Cleaned/DB1B_Market_model_ready.parquet
Shape:  (7189747, 15)
